# Pipeline ETL: Integración Mensual del Mercado del Arroz en Colombia (2018–2026)
**Equipo 10:** Sebastián Cardona Franco & Miguel Ángel Echavarría

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Consolidación y Filtrado de SIPSA-DANE (Arroz Comercial en Grano)

In [ ]:
variedades_arroz_comercial = [
    'ARROZ DE PRIMERA',
    'ARROZ DE SEGUNDA',
    'ARROZ EXCELSO',
    'ARROZ BLANCO IMPORTADO',
    'ARROZ BLANCO EMPACADO',
    'ARROZ BLANCO EN BULTO'
]

archivos_sipsa = sorted(glob.glob('../01_datos_fuentes/anex-SIPSA-*.xlsx') + glob.glob('../01_datos_fuentes/series-historicas-*.xlsx'))
if archivos_sipsa:
    lista_df = []
    for archivo in archivos_sipsa:
        xls = pd.ExcelFile(archivo)
        for hoja in xls.sheet_names:
            if hoja.strip().upper() in ['ÍNDICE', 'INDICE', 'FUENTES', 'CPC']:
                continue
            df_tmp = pd.read_excel(archivo, sheet_name=hoja, header=None, nrows=15)
            fila_enc = 0
            for idx, row in df_tmp.iterrows():
                txt = ' '.join([str(v).upper() for v in row.values if pd.notna(v)])
                if 'FECHA' in txt and ('PRODUCTO' in txt or 'PRECIO' in txt):
                    fila_enc = idx
                    break
            df_h = pd.read_excel(archivo, sheet_name=hoja, skiprows=fila_enc)
            df_h.columns = [str(c).strip().upper() for c in df_h.columns]
            col_p = [c for c in df_h.columns if 'PRECIO' in c][0]
            df_h = df_h.rename(columns={col_p: 'PRECIO'})
            sub = df_h[['FECHA', 'PRODUCTO', 'PRECIO']].dropna().copy()
            sub['FECHA'] = pd.to_datetime(sub['FECHA'], errors='coerce')
            sub['PRECIO'] = pd.to_numeric(sub['PRECIO'], errors='coerce')
            sub['PRODUCTO'] = sub['PRODUCTO'].astype(str).str.strip().str.upper()
            sub = sub[sub['PRODUCTO'].isin(variedades_arroz_comercial)].dropna()
            lista_df.append(sub)
    df_sipsa_raw = pd.concat(lista_df, ignore_index=True)
    df_sipsa_raw['PERIODO'] = df_sipsa_raw['FECHA'].dt.strftime('%Y-%m-01')
    df_sipsa_raw['precio_arroz_cop_ton'] = df_sipsa_raw['PRECIO'] * 1000.0
    df_sipsa = df_sipsa_raw.groupby('PERIODO', as_index=False)['precio_arroz_cop_ton'].mean()
else:
    df_sipsa = pd.read_csv('../01_datos_fuentes/Fuente2_Secundaria_SIPSA_DANE_Arroz.csv')[['PERIODO', 'precio_arroz_cop_ton']]

df_sipsa.head()

## 2. Procesamiento de Cotizaciones Internacionales (Banco Mundial - The Pink Sheet)

In [ ]:
if os.path.exists('../01_datos_fuentes/CMO-Historical-Data-Monthly.xlsx'):
    df_ps = pd.read_excel('../01_datos_fuentes/CMO-Historical-Data-Monthly.xlsx', sheet_name='Monthly Prices', skiprows=4)
    df_ps = df_ps.rename(columns={df_ps.columns[0]: 'PERIODO_RAW'})
    cols_arroz = ['Rice, Thai 5%', 'Rice, Thai 25%', 'Rice, Thai A.1', 'Rice, Viet Namese 5%']
    df_ps = df_ps[df_ps['PERIODO_RAW'].astype(str).str.match(r'^\d{4}M\d{2}$')].copy()
    df_ps['PERIODO'] = df_ps['PERIODO_RAW'].str.slice(0, 4) + '-' + df_ps['PERIODO_RAW'].str.slice(5, 7) + '-01'
    for c in cols_arroz:
        df_ps[c] = pd.to_numeric(df_ps[c], errors='coerce')
    df_ps = df_ps[(df_ps['PERIODO'] >= '2018-01-01') & (df_ps['PERIODO'] <= '2026-10-01')]
    df_ps['precio_arroz_usd_ton'] = df_ps[cols_arroz].mean(axis=1)
    df_pink = df_ps[['PERIODO', 'precio_arroz_usd_ton']].copy()
else:
    df_pink = pd.read_csv('../01_datos_fuentes/Fuente3_Internacional_PinkSheet_BancoMundial.csv')[['PERIODO', 'precio_arroz_usd_ton']]

df_pink.head()

## 3. Agregación Mensual de la Fuente Primaria (Sensor Agroclimático Simulado)

In [ ]:
df_sensor_raw = pd.read_csv('../01_datos_fuentes/Fuente1_Primaria_Sensor_Agroclimatico.csv')
df_sensor_raw['PERIODO'] = df_sensor_raw['fecha'].astype(str).str.slice(0, 7) + '-01'
df_sensor = (
    df_sensor_raw.groupby('PERIODO', as_index=False)
    .agg(
        precipitacion_mm=('precipitacion_mm', 'sum'),
        temperatura_C=('temperatura_C', 'mean'),
        humedad_relativa_pct=('humedad_relativa_pct', 'mean')
    )
    .sort_values('PERIODO')
)
df_sensor.head()

## 4. Integración Relacional (MERGE) y Cálculo de Retorno Logarítmico, IPAt y Volatilidad Vt

In [ ]:
df_integrado = (
    df_sipsa.merge(df_pink, on='PERIODO', how='inner')
    .merge(df_sensor, on='PERIODO', how='inner')
    .sort_values('PERIODO')
    .reset_index(drop=True)
)

df_integrado['retorno_log'] = np.log(
    df_integrado['precio_arroz_cop_ton'] / df_integrado['precio_arroz_cop_ton'].shift(1)
)
df_integrado['volatilidad_Vt'] = df_integrado['retorno_log'].rolling(window=6).std()

cg3 = (df_integrado['precio_arroz_cop_ton'] / df_integrado['precio_arroz_cop_ton'].shift(3)) ** (1 / 3) - 1
df_integrado['IPAt'] = (cg3 - cg3.rolling(24, min_periods=6).mean()) / cg3.rolling(24, min_periods=6).std()

columnas_finales = [
    'PERIODO',
    'precipitacion_mm',
    'temperatura_C',
    'humedad_relativa_pct',
    'precio_arroz_cop_ton',
    'precio_arroz_usd_ton',
    'retorno_log',
    'IPAt',
    'volatilidad_Vt'
]
df_integrado = df_integrado[columnas_finales]
df_integrado.to_csv('Arroz_Integrado_2018_2026.csv', index=False, encoding='utf-8-sig')
print('Dimensiones finales:', df_integrado.shape)
df_integrado.head(10)